FRAME TO FRAME ALL AUDIO

In [1]:
# imports from run_test
from __future__ import print_function
from __future__ import division
import time, os, sys
import numpy as np
import soundfile as sf
from spicy import signal
import pickle
import warnings
import gzip
import scipy.io
from scipy.io import wavfile
from scipy.io import loadmat
import matplotlib.pyplot as plt

warnings.simplefilter('ignore')

import torch
import torch.utils.data as data

#workspace_dir = '/home/adelval/BTS/TFM/afterburner8k/'
workspace_dir = '/home/adelval/BTS/TFM/test/'

sys.path.append(workspace_dir + 'src/net')
sys.path.append(workspace_dir + 'src/train')
sys.path.append(workspace_dir + 'src/eval')
from vvtk_net.v1.utils import *
from vvtk_net.v1.transforms import *
from vvtk_net.v1.transforms_fe import *
from vvtk_net.v1.datafeed import *
from vvtk_net.v1.layers_pytorch import *
from vvtk_net.config import Configuration
from eval_utils import *

In [2]:

def read_audio(f):
    x, fs = sf.read(f)
    return np.array(x * 2**15, dtype=np.int16), fs

In [3]:

def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)

def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x

# Divide x into overlapping frames of fixed length
def windowing2(x, fs=16000, Ns=0.025, Ms=0.010):
    N = int(Ns * fs)            # Number of samples in each window
    M = int(Ms * fs)            # Step size (number of samples between window starts)
    n = (len(x) + M - 1) // M   # Number of frames
    print(len(x))
    # print("Number of shifts", n)    
    T = (n - 1) * M + N         # Total signal length needed to fit the frames
    print(T)
    xa = x.copy()
    if T > len(x):
        xa.resize(T, refcheck=False)    # Pad the signal with zeros
    m = np.arange(0, n * M, M)          # Starting indices of each frame
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1) # 2D matrix whit rows correspnding to the samples whitin a window and columns the starting indeces each time
    return xa[ind.astype(int).T].astype(np.float32)

def hamming(X):
    w = np.hamming(X.shape[1])
    return X * w

def fft(X, NFFT=None):
    if NFFT is None:
        NFFT = int(2 ** np.ceil(np.log2(X.shape[0])))
    Xfreq = np.abs(np.fft.fft(X, NFFT, axis=0)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals


def frame_fft(data, fs, w, nfft):
    
    N =[int(wi * fs) for wi in w]
    F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]

    x = offset(data)

    # Emphasis to increase the amplitude of high freq
    x = preemphasis(x)

    print(f'0 Window frames {x[:,:10]}')

    XX = []

    X = hamming(x)
    # print("El frame enventanado tiene dimensión: ",X.shape)
    # Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
    Xfft = fft(X, nfft[0])
    X = Xfft * Xfft # Power spectrum
    X = np.asarray(X, dtype=np.float32)
    # print("Vector con Power Spectral Density del frame: \n", X[:10])

    return X    # Returns the PSD of the frame processed


# Log scale for PSD
def log_scale(frame_psd):
    scale=1.
    eps=1e-8
    x = frame_psd    
    x = np.abs(x)
    x = scale * np.log10(x + eps)  
    # print("Vector con Power Spectral Density in log scale del frame: \n", x[:10])

    return x   


In [4]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients
def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb

def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b


def frame_fb_mfcc(data, fs, B, w, nfft):

    N =[int(wi * fs) for wi in w]
    F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]
    fb = [ fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]
    #print(f'fb es {len(fb[0])}')
    dct = [ f_base_dct(Bi) for Bi in B] 
    #print(f'dct es {len(dct[0])}')


    x = offset(data)
    x = preemphasis(x)
    XX = []
    for i,w in enumerate(w):
        X = hamming(x)
        
        Xfft = fft(X, nfft[i])
        Xb = np.log(Xfft.dot( fb[i] ) + 1)
        Xc = Xb.dot(dct[i])                                
        
        X = np.concatenate( [Xb, Xc], 0 )
        
        X = np.asarray(X, dtype=np.float32)
        XX.append(X)
    XX = np.concatenate(XX, 0)
    
    #print("El tamaño de x08k2 es: ",XX.shape)
    print("Vector con FilterBank MFCC del frame: \n", XX[:10])
    
    return XX

def read_pkl(f):
    with open(f, 'rb') as file:  # Open the file in binary read mode
        return pickle.load(file)

def norm_fb_frame(frame_fbmfcc):
    # Normalization of fbmfcc
    file = workspace_dir + 'data/model/fe1_norm1.pkl'  # de donde salen??
    x = frame_fbmfcc
    mu, std = read_pkl(file)
    x -= mu
    x /= std + 1e-6
    # print("Vector con FB MFCC normalizado del frame: \n", x[:10])

    return x
    

In [5]:
# Load model dimensions and weights

def load_obj(file):
    if not isinstance(file,str):
        return pickle.load(f)

    root,ext = os.path.splitext(file)
    if ext == '.gz':
        with gzip.open(file, 'rb') as f:
            return pickle.load(f)
    else:
        with open(file, 'rb') as f:
            return pickle.load(f)

input_dim, output_dim = load_obj(workspace_dir + 'data/model/dimensions.pkl') 

print('  input_dim: %s' % str(input_dim))
print('  output_dim: %s' % str(output_dim))


import sys
sys.path.append( workspace_dir + 'src/net')

from net_snr import Net_snr

net_snr = Net_snr(input_dim, output_dim, cuda=False)
net_snr.load_theta( workspace_dir + 'data/model/theta_last')

  input_dim: 576
  output_dim: 512

  Net_snr:


    nb_params: 29.99M
    cuda: False
    float16: False
    single_gpu: True
    opt: adama, 0.001, None
Adam (
Parameter Group 0
    amsgrad: True
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0
)
    reading obj /home/adelval/BTS/TFM/test/data/model/theta_last


In [6]:
# Model inference
def net_eval(frame_concat):

    net_snr.set_mode_train(False)

    x = frame_concat
    print(x.shape)
    # x = x.reshape(1, -1)
    # print(x.shape)

    snr = net_snr.predict(x)
    print(snr[0,:])
    snr = to_numpy(snr.squeeze())
    print(f'La máscara del frame caculado es de {snr.shape}')
    #scipy.io.savemat(f, mdict={'snr': snr})
    x = to_numpy(x.squeeze())
    snr = to_numpy(snr.squeeze())
    return snr


In [7]:
# Aplicar mascara a frame 

def apply_filter(data, filt, frame=640, shift=160, nfft=1024):

    win = np.sqrt(np.hanning(frame))
    win = np.array(win, dtype=np.float32)
    
    yw = np.zeros(data.size)
    print(data.size)
    #yw[0:frame] = data[0:frame]*win*1e-3
    #it = int(np.floor((data.size-frame)/shift))
    #print(f'La ventana se desplazara {it} veces')
    it = 1
    for i in range(0,it):
        print(f'El frame sin enventanado con rn resulta {data[:10]}')
        xw = data[i*shift : i*shift+frame] * win
        print(f'El frame enventanado resulta {xw[:10]}')
        Xfft = np.fft.fft(xw, nfft)
        Xfft = Xfft[0:int(nfft/2+1)]
        print(Xfft.shape)
        print(f'El tamaño de la transformada del frame es {Xfft.shape}')
        print(f'La transformada del frame es {Xfft[:10]}')
        outf = Xfft * filt[:,i]
        print(f'El tamaño de la salida del filtro sera {outf.shape}')
        print(f'La salida del filtro es {outf[:10]}')
        fliped = np.flip(np.conj(outf[1:int(nfft/2)]), axis=0)
        outw = np.concatenate((outf, fliped), axis=0)
        outw = np.real(np.fft.ifft(outw, nfft, axis=0))
        print(f'El frame mejorado resulta sin OLA es {outw[:10]}')
        #print(f'La salida de la ifft tendra {outw.shape} samples') # de las que nos quedamos con 640 porque el resto son relleno
        yw[i*shift : i*shift+frame] = yw[i*shift : i*shift+frame] + outw[0:frame]*win
        print(f'El frame mejorado resulta {yw[:10]}')

    #yw[it*shift+frame:] = data[it*shift+frame:]*1e-3

    return yw

# ----------------------------------------------------------------------------------------------------------------------
def noiseReduction(data, snr_net, fs, frame, shift, nfft, gmin):
    
    # Add un-audible noise to avoid signals with 0
    data = data + 1e-7*np.random.rand(data.size) 
    print(snr_net.shape)
    snr_net = snr_net.reshape(-1,1)
    print("2",snr_net.shape)
    # Load snr_net
    snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]), axis=0)
    #snr_net = np.concatenate((snr_net, snr_net[int(nfft/2-1):, :]))
    # VAD
    ini = int(np.floor((nfft*300)/fs))
    out = int(np.ceil((nfft*2500)/fs))
    E = np.mean(snr_net[ini:out,:], axis=0)
    vad = 1/(1 + np.exp(-90*(E-0.05)))

    difference = 1 - snr_net
    difference[difference < 1e-7] = 1e-7
    gamma = 1 / difference
    upsilon = gamma * snr_net
    g = np.exp(0.5 * sc.exp1(upsilon)) * snr_net
    g[g > 1] = 1
    gmin = gmin/2
    gtotal = np.power(g, snr_net) * np.power((gmin * snr_net), (1 - snr_net))
    filt = np.power(gtotal,vad) * np.power(1e-3,1-vad)
    print(f'El filtro será {filt[:10]}')
    yw = apply_filter(data, filt, frame, shift, nfft)

    return yw, filt


In [8]:
x_test = ['/home/adelval/BTS/TFM/audios/audio_1.wav']
#x_test = ['/home/adelval/BTS/TFM/audios/7-CH0_C01_city_5dB.wav']
#x_test = ['/home/adelval/BTS/TFM/afterburner8k/data/audio/minitest_8k/5-CH0_C01_stadium_15dB.wav']
print('  x_test: %s' % (x_test))

  x_test: ['/home/adelval/BTS/TFM/audios/audio_1.wav']


In [9]:
audio, fs = read_audio(x_test[0])
print(f'La frecuencia de muestreo es {fs} Hz')
print(f'La duración del audio es {len(audio)/fs} segundos y {len(audio)} muestras')

# Parámetros
#fs=16000
B=[32]
w=[0.040]
m=0.010
nfft=[1024]
gmin = 0.0562


frame_size = 0.01  # 10 ms
frame_samples = int(frame_size * fs)  # Muestras por frame
print(f'Un frame tiene una duración de {frame_samples} samples')
shift_size = 0.01  # 10 ms
shift_samples = int(shift_size * fs)  # Desplazamiento entre frames 160
print(f'Desplazamiento de {shift_samples} samples')
window_size = 0.04  # 40 ms (640 muestras)
window_samples = int(window_size * fs)  # Muestras por ventana 640
print(f'La ventana tiene una duración de {window_samples} samples')
max_window = 20
window_inference = int((max_window + (window_size/frame_size)-1) * frame_samples)
print(window_inference)
buffer_frame = np.zeros(0)  # Buffer de ventana recibida
# CALCULO DE LA MÁSCARA SNR 
for n_frame in range(23):
    # Obtener el frame de audio
    frame = audio[n_frame * shift_samples: n_frame * shift_samples + frame_samples]
    
    # Concatenar el frame recibido al buffer `buffer_frame`
    buffer_frame = np.concatenate([buffer_frame, frame])
    # Si el buffer alcanza o excede las 20 ventanas para hacer la inferencia
    if len(buffer_frame) >= window_inference:
        print("Reached")
        work_inf_frames = buffer_frame[:window_inference]
print(f'0 frame  {work_inf_frames[:10]}')
print(f'1 frame  {work_inf_frames[frame_samples:frame_samples+10]}')
print(f'2 frame  {work_inf_frames[2*frame_samples:2*frame_samples+10]}')
print(f'3 frame  {work_inf_frames[3*frame_samples:3*frame_samples+10]}')
print(f'23 frame {work_inf_frames[22*frame_samples:22*frame_samples+10]}')


La frecuencia de muestreo es 16000 Hz
La duración del audio es 4.655 segundos y 74480 muestras
Un frame tiene una duración de 160 samples
Desplazamiento de 160 samples
La ventana tiene una duración de 640 samples
3680
Reached
0 frame  [ 0. -1. -1.  0. -1.  0.  0.  0.  1.  1.]
1 frame  [-1.  0.  0.  0.  0.  0.  0. -1. -1.  0.]
2 frame  [ 0.  0.  0.  0.  0.  1.  0. -1. -1.  0.]
3 frame  [ 0.  0. -1.  0.  0.  0.  0.  0.  0.  1.]
23 frame [ 1.  0.  1.  0.  0.  0.  1.  0. -1.  0.]


In [10]:
#FFT
from spicy import signal

def offset(x):
    return signal.lfilter([1.0, -1.0],[1, -0.99899,], x)

def preemphasis(x):
    #return signal.lfilter([1.0, -0.97],1, x)    
    x0 = x[0] * (1.0 - 0.97)
    for i in reversed(range(len(x))): 
        x[i] = x[i] - x[i-1] * 0.97
    x[0] = x0
    return x

# Divide x into overlapping frames of fixed length without extending to slide last frame
def windowing2(x, fs=16000, Ns=0.025, Ms=0.010, Mw=20):
    # Mw limit the number of windows
    N = int(Ns * fs)            # Number of samples in each window 640
    M = int(Ms * fs)            # Step size (number of samples between window starts) 160
    n = (len(x) + M - 1) // M   # Number of frames 23
    print("Number of frames", n)    
    # T = (n - 1) * M + N         # Total signal length needed to fit the frames
    xa = x.copy()
    # Se ignora el padding porque la ventana se deslizará
    # if T > len(x):
    #     print("rellena con ceros")
    #     xa.resize(T, refcheck=False)    # Pad the signal with zeros
    m = np.arange(0, Mw*M, M)          # Starting indices of each frame
    print("mshape",m.shape)
    print(m)
    ind = np.arange(N).reshape(-1, 1) + m.reshape(1, -1) # 2D matrix whit rows correspnding to the samples whitin a window and columns the starting indeces each time
    print(ind.shape)
    print(ind)
    xa = xa[ind.astype(int).T].astype(np.float32)
    # print(f'Window frames {xa[19,:5]}')

    return xa

def hamming(X):
    print("la de X ", X.shape)
    w = np.hamming(X.shape[1])
    return X * w

def fft(X, NFFT=None):
    if NFFT is None:
        NFFT = int(2 ** np.ceil(np.log2(X.shape[1])))
    Xfreq = np.abs(np.fft.fft(X, NFFT, axis=1)) # Computes the FFT along each row of X
    Xfreq = np.asarray(Xfreq, dtype=np.float32)
    return Xfreq[:, :(NFFT // 2)] # Return Half the Spectrum, only the first half is meaningful for real-valued signals


N =[int(wi * fs) for wi in w]
F =[int(2 ** np.ceil(np.log2( Ni)))//2 for Ni in N]


x = offset(work_inf_frames)
# Emphasis to increase the amplitude of high freq
x = preemphasis(x)
print(f'0 frame   {x[:10]}')
print(f'1 frame   {x[1*int(m*fs):1*int(m*fs)+10]}')
print(f'2 frame   {x[2*int(m*fs):2*int(m*fs)+10]}')
print(f'3 frame   {x[3*int(m*fs):3*int(m*fs)+10]}')
print(f'20 frame  {x[19*int(m*fs):19*int(m*fs)+10]}')



X = hamming(windowing2(x, fs=fs, Ns=w[0], Ms=m, Mw=max_window))
# Dimesions of X are (N,n) having in row the signal windowed with hamming of win length
Xfft = fft(X, nfft[0])
print(f"la shape de Xfft es {Xfft.shape}")
X = Xfft * Xfft # Power spectrum
X = np.asarray(X, dtype=np.float32)

print("0 Vector 2D con FFT para cada frame por row \n",  X[0,:10])
print("1 Vector 2D con FFT para cada frame por row \n",  X[1,:10])
print("2 Vector 2D con FFT para cada frame por row \n",  X[2,:10])
print("3 Vector 2D con FFT para cada frame por row \n",  X[3,:10])
print("20 Vector 2D con FFT para cada frame por row \n", X[19,:10])

# Log escale
fft_windows_log = log_scale(X)
print(" 0 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[0,:10])
print(" 1 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[1,:10])
print(" 2 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[2,:10])
print(" 3 Vector 2D con FFT en escala log para cada frame por row \n",  fft_windows_log[3,:10])
print("20 Vector 2D con FFT en escala log para cada frame por row \n", fft_windows_log[19,:10])

0 frame   [ 0.00000000e+00 -1.00000000e+00 -2.89900000e-02  9.71039280e-01
 -9.99941470e-01  9.71068471e-01  8.76919559e-05  8.76033871e-05
  1.00008751e+00  2.90774265e-02]
1 frame   [-1.00007831e+00  9.70931768e-01 -4.88727982e-05 -4.88234367e-05
 -4.87741250e-05 -4.87248632e-05 -4.86756510e-05 -1.00004863e+00
 -2.90385774e-02  9.70990752e-01]
2 frame   [-9.70854401e-01  1.26161699e-04  1.26034276e-04  1.25906981e-04
  1.25779815e-04  1.00012565e+00 -9.70884474e-01 -9.99903881e-01
 -2.88939779e-02  9.71135205e-01]
3 frame   [-4.07759293e-05 -4.07347456e-05 -1.00004069e+00  9.70969347e-01
 -1.13315439e-05 -1.13200991e-05 -1.13086658e-05 -1.12972440e-05
 -1.12858338e-05  9.99988726e-01]
20 frame  [1.14859746e-04 1.14743738e-04 1.14627847e-04 1.14512073e-04
 1.14396416e-04 1.14280875e-04 1.14165452e-04 1.14050144e-04
 1.13934954e-04 1.00011382e+00]
Number of frames 23
mshape (20,)
[   0  160  320  480  640  800  960 1120 1280 1440 1600 1760 1920 2080
 2240 2400 2560 2720 2880 3040]
(640

In [11]:
#FB MFCC Filter Bank Mel-Frequency Cepstral Coefficients

def fb_etsi(F, B, fs):
    StFreq = 64.0
    fb = np.zeros((F, B))
    # /* Constants for calculation*/
    start_mel = 2595.0 * np.log10(1.0 + StFreq / 700)
    fs_per_2_mel = 2595.0 * np.log10(1.0 + (fs / 2) / 700)
    for b in range(B):
        # /* Calculating mel-scaled frequency and the corresponding FFT-bin */
        # /* number for the lower edge of the band                          */
        freq = 700 * (np.power(10.0, (start_mel + (b) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        f1 = (2 * F * freq / fs + 0.5)
        # /* Calculating mel-scaled frequency for the upper edge of the band */
        freq = 700 * (
        np.power(10.0, (start_mel + (b + 2) / (B + 1) * (fs_per_2_mel - start_mel)) / 2595) - 1.0)
        # /* Calculating and storing the length of the band in terms of FFT-bins*/
        f3 = (2 * F * freq / fs + 0.5)
        f2 = (f1 + f3) / 2
        f3 = min(f3, F - 1)
        s = 0.0
        f1 = int(f1)
        f2 = int(f2)
        f3 = int(f3)
        for f in range(f1, f2):
            fb[f, b] = f * (1 / (f2 - f1)) - f1 / (f2 - f1)
            s += fb[f, b]
        for f in range(f2, f3):
            fb[f, b] = f * (-1 / (f3 - f2)) + f3 / (f3 - f2)
            s += fb[f, b]
        for f in range(f1, f3):
            fb[f, b] /= s  # //normalization
    return fb

def f_base_dct(N):
    b = np.zeros((N, N))
    for n in range(N):
        if n == 0:
            kn = np.sqrt(1 / N)
        else:
            kn = np.sqrt(2 / N)
        for m in range(N):
            b[m, n] = kn * np.cos((2 * m + 1) * n * np.pi / (2 * N))
    return b

fb = [ fb_etsi(Fi, Bi, fs) for Fi, Bi in zip(F, B)]
print(f'fb es {len(fb[0])}')
dct = [ f_base_dct(Bi) for Bi in B] 
print(f'dct es {len(dct[0])}')

x = offset(work_inf_frames)
# Emphasis to increase the amplitude of high freq
x = preemphasis(x)

X = hamming(windowing2(x, fs=fs, Ns=w[0], Ms=m, Mw=20))

Xfft = fft(X, nfft[0])
Xb = np.log(Xfft.dot( fb[0] ) + 1)
Xc = Xb.dot(dct[0])

X = np.concatenate( [Xb, Xc], 1 )

X = np.asarray(X, dtype=np.float32)

print(X.shape)
print(f'0 Filtro Mel {X[0,:5]}')
print(f'1 Filtro Mel {X[1,:5]}')
print(f'2 Filtro Mel {X[2,:5]}')
print(f'3 Filtro Mel {X[3,:5]}')
print(f'19 Filtro Mel {X[19,:5]}')

file = workspace_dir + 'data/model/fe1_norm1.pkl' 
mu, std = read_pkl(file)
fbmfcc_window_norm = norm_fb_frame(X)
print(f'0 Filtro Mel normalizado {fbmfcc_window_norm[0,:5]}')
print(f'1 Filtro Mel normalizado {fbmfcc_window_norm[1,:5]}')
print(f'2 Filtro Mel normalizado {fbmfcc_window_norm[2,:5]}')
print(f'3 Filtro Mel normalizado {fbmfcc_window_norm[3,:5]}')
print(f'19 Filtro Melnormalizado {fbmfcc_window_norm[19,:5]}')

fb es 512
dct es 32
Number of frames 23
mshape (20,)
[   0  160  320  480  640  800  960 1120 1280 1440 1600 1760 1920 2080
 2240 2400 2560 2720 2880 3040]
(640, 20)
[[   0  160  320 ... 2720 2880 3040]
 [   1  161  321 ... 2721 2881 3041]
 [   2  162  322 ... 2722 2882 3042]
 ...
 [ 637  797  957 ... 3357 3517 3677]
 [ 638  798  958 ... 3358 3518 3678]
 [ 639  799  959 ... 3359 3519 3679]]
la de X  (20, 640)
(20, 64)
0 Filtro Mel [0.23243642 0.28519294 0.5673354  0.683461   0.7875668 ]
1 Filtro Mel [0.21219873 0.25336957 0.5756598  0.7295447  0.95927125]
2 Filtro Mel [0.21223295 0.25848418 0.41061014 0.50089383 0.77805275]
3 Filtro Mel [0.34777853 0.36755627 0.45605582 0.41130796 0.7351749 ]
19 Filtro Mel [0.34695724 0.57377666 0.37493095 0.4249149  0.5653557 ]
0 Filtro Mel normalizado [-2.984838  -2.8981009 -2.7997372 -2.7685244 -2.7185802]
1 Filtro Mel normalizado [-2.9946365 -2.9120762 -2.796107  -2.7484264 -2.6457765]
2 Filtro Mel normalizado [-2.9946198 -2.90983   -2.8680854 -2.8

In [12]:
print(f'Tamaño de fft es {fft_windows_log.shape} y de fbmfcc es {fbmfcc_window_norm.shape}')
windows_concat = np.concatenate( (fft_windows_log,fbmfcc_window_norm), 1 )
print("El tamaño de x08k tras la concatenacion es: ",windows_concat.shape)
print(f'0 La concatenacion resulta  {windows_concat[0,:5]}  y {windows_concat[0,512:517]}')
print(f'1 La concatenacion resulta  {windows_concat[1,:5]}  y {windows_concat[1,512:517]}')
print(f'2 La concatenacion resulta  {windows_concat[2,:5]}  y {windows_concat[2,512:517]}')
print(f'3 La concatenacion resulta  {windows_concat[3,:5]}  y {windows_concat[3,512:517]}')
print(f'19 La concatenacion resulta {windows_concat[19,:5]} y {windows_concat[19,512:517]}')

Tamaño de fft es (20, 512) y de fbmfcc es (20, 64)
El tamaño de x08k tras la concatenacion es:  (20, 576)
0 La concatenacion resulta  [-2.47386    -1.5200697  -1.0509288  -0.8417688  -0.70806146]  y [-2.984838  -2.8981009 -2.7997372 -2.7685244 -2.7185802]
1 La concatenacion resulta  [-1.5796448 -1.5199704 -1.7905464 -1.2009634 -0.7062188]  y [-2.9946365 -2.9120762 -2.796107  -2.7484264 -2.6457765]
2 La concatenacion resulta  [-1.3838824 -1.1336191 -1.7873496 -2.563182  -1.5647051]  y [-2.9946198 -2.90983   -2.8680854 -2.8481464 -2.7226143]
3 La concatenacion resulta  [-1.0911639  -1.4584094  -1.1114112  -0.90263665 -0.8511954 ]  y [-2.9289918 -2.861931  -2.8482666 -2.8872168 -2.740795 ]
19 La concatenacion resulta [-1.3306018 -1.2445422 -1.3499471 -1.991468  -3.9184818] y [-2.9293892 -2.771369  -2.8836453 -2.8812828 -2.8127995]


In [13]:
snr_frame_mask = net_snr.predict(windows_concat)


torch.Size([1, 20, 576])
del bloque 1 torch.Size([1, 512, 20])
SGE prev 0 tensor([[[-2.0481e-01, -5.6786e-01, -4.8141e-01,  ...,  5.4534e-01,
           6.7702e-01,  4.5562e-01],
         [-2.9722e-01, -3.4928e-01, -6.1219e-01,  ..., -6.0560e-01,
          -6.3185e-01, -5.8202e-01],
         [ 3.7734e-01,  3.3345e-02,  1.4872e-01,  ...,  2.9345e-01,
          -1.8623e-01, -2.4870e-01],
         ...,
         [-9.4883e-01, -6.9302e-01, -1.0131e+00,  ..., -7.8467e-01,
           1.2471e-01,  1.1285e+00],
         [-1.6190e-01, -1.1914e-01,  1.0383e-01,  ...,  6.4062e-01,
           2.4854e-01,  3.8642e-01],
         [ 5.8899e-01,  1.5080e-01,  1.3234e-01,  ..., -3.1408e-02,
           9.2074e-01, -2.6359e-01]],

        [[-5.5310e-01, -4.4793e-01, -1.2214e-01,  ..., -5.3082e-01,
          -3.0479e-01, -1.3955e+00],
         [-1.4103e-01, -2.0430e-02,  9.2541e-02,  ...,  2.7214e-02,
           1.0143e-01, -9.1063e-01],
         [-1.1462e-01, -1.1642e-01, -4.4969e-01,  ..., -4.5837e-01,
  

In [14]:
snr_frame_mask = to_numpy(snr_frame_mask.squeeze())

print(snr_frame_mask.shape)
print(snr_frame_mask[0,:10])
print(snr_frame_mask[1,:10])
print(snr_frame_mask[2,:10])
print(snr_frame_mask[3,:10])
print(snr_frame_mask[19,:10])

(20, 512)
[0.98993814 0.9991295  0.9999058  0.999979   0.99998784 0.99998474
 0.9999863  0.99846894 0.9617356  0.9856248 ]
[0.9487386  0.9437837  0.93137574 0.92023414 0.9194411  0.938006
 0.9638132  0.9704845  0.96846575 0.97544086]
[0.907254   0.86166924 0.777209   0.7486785  0.76966393 0.8095259
 0.85988986 0.9035461  0.93516994 0.9581362 ]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[0.29834044 0.5547544  0.6178942  0.70191365 0.83110833 0.8135099
 0.81185997 0.8019167  0.623319   0.8514687 ]
